In [28]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
importlib.reload(kinematics)
from kinematics import azaltroll_to_theta,apply_mechanical_corrections, q_from_azaltroll, MountModelParams


In [29]:
csv_filename = './test_validate.csv'
d = pd.read_csv(csv_filename)
d.describe()
d['dev_p_theta1'] = ((d['s_theta1'] - d['p_theta1'] + 180) % 360 - 180)*60
d['dev_p_theta2'] = ((d['s_theta2'] - d['p_theta2'] + 180) % 360 - 180)*60
d['dev_p_theta3'] = ((d['s_theta3'] - d['p_theta3'] + 180) % 360 - 180)*60
d['dev_m_theta1'] = ((d['s_theta1'] - d['m_theta1'] + 180) % 360 - 180)*60
d['dev_m_theta2'] = ((d['s_theta2'] - d['m_theta2'] + 180) % 360 - 180)*60
d['dev_m_theta3'] = ((d['s_theta3'] - d['m_theta3'] + 180) % 360 - 180)*60
d['g_az'] = np.round(d['p_az'] / 5) * 5 
d['g_alt'] = np.round(d['p_alt'] / 5) * 5
d['g_roll'] = np.round(d['p_roll'] / 5) * 5
d.columns

Index(['session_id', 'filename', 'status', 'date_obs', 'site_lat', 'site_lon',
       'p_az', 'p_alt', 'p_roll', 'p_theta1', 'p_theta2', 'p_theta3', 's_az',
       's_alt', 's_roll', 's_theta1', 's_theta2', 's_theta3', 'dev_p_az',
       'dev_p_alt', 'dev_p_roll', 'pixel_scale_arcsec', 'sync_point',
       'alignQ_w', 'alignQ_x', 'alignQ_y', 'alignQ_z', 'm_az', 'm_alt',
       'm_roll', 'm_theta1', 'm_theta2', 'm_theta3', 'dev_m_az', 'dev_m_alt',
       'dev_m_roll', 'dev_p_theta1', 'dev_p_theta2', 'dev_p_theta3',
       'dev_m_theta1', 'dev_m_theta2', 'dev_m_theta3', 'g_az', 'g_alt',
       'g_roll'],
      dtype='object')

In [30]:
d[['m_az', 'm_alt', 'm_roll', 'm_theta1', 'm_theta2', 'm_theta3','dev_m_theta1','dev_m_theta2','dev_m_theta3'  ]].corr(numeric_only=True)

,m_az,m_alt,m_roll,m_theta1,m_theta2,m_theta3,dev_m_theta1,dev_m_theta2,dev_m_theta3
m_az,1.000000,-0.041306,0.054824,0.417642,-0.061762,-0.044670,0.125793,0.029672,-0.239063
m_alt,-0.041306,1.000000,0.035610,-0.009345,0.958111,-0.088128,0.368316,0.175313,-0.239512
m_roll,0.054824,0.035610,1.000000,-0.048766,-0.033102,-0.911463,0.275805,0.368696,-0.059603
m_theta1,0.417642,-0.009345,-0.048766,1.000000,-0.005226,0.039370,0.190398,0.138510,-0.372321
m_theta2,-0.061762,0.958111,-0.033102,-0.005226,1.000000,-0.010775,0.185266,0.077417,-0.087401
m_theta3,-0.044670,-0.088128,-0.911463,0.039370,-0.010775,1.000000,-0.332218,-0.398928,0.050745
dev_m_theta1,0.125793,0.368316,0.275805,0.190398,0.185266,-0.332218,1.000000,0.191914,-0.908069
dev_m_theta2,0.029672,0.175313,0.368696,0.138510,0.077417,-0.398928,0.191914,1.000000,-0.114440
dev_m_theta3,-0.239063,-0.239512,-0.059603,-0.372321,-0.087401,0.050745,-0.908069,-0.114440,1.000000


# Review of Collected Data and Residuals

In [31]:
fig = go.Figure()
legend_offset = 400
d['legend_az'] = d['p_az']/360*100 + legend_offset
d['legend_alt'] = d['p_alt'] + legend_offset
d['legend_roll'] = d['p_roll'] + legend_offset
xfield='date_obs'
for yfield in ['dev_p_az','dev_p_alt','dev_p_roll', 'legend_az', 'legend_alt', 'legend_roll']:
    fig.add_trace(go.Scatter(
        x=d[xfield], y=d[yfield], 
        mode='markers+lines', name=yfield,
        marker=dict(size=6),
    ))

fig.update_layout(
    title=dict(text=f'Axis Residuals vs {xfield}', x=0.5, font=dict(size=24, family='Arial')),
    xaxis_title=f'{xfield}', yaxis_title=f'Axis Residuals (arc-min)', 
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    hovermode='x unified',
    height=800, width=1400
)

fig.show()

# Theta Space Residuals - BEFORE

In [32]:
fig = px.scatter_matrix(d, dimensions=["dev_p_theta1", "dev_p_theta2", "dev_p_theta3", "p_theta1", "p_theta2", "p_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Theta Space Residuals - AFTER

In [33]:
fig = px.scatter_matrix(d, dimensions=["dev_m_theta1","dev_m_theta2", "dev_m_theta3", "m_theta1", "m_theta2", "m_theta3"], color="m_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - BEFORE

In [38]:
fig = px.scatter_matrix(d, dimensions=["dev_p_az", "dev_p_alt", "dev_p_roll", "p_theta1", "p_theta2", "p_theta3"], color="p_alt")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - AFTER

In [8]:
fig = px.scatter_matrix(d, dimensions=["dev_m_az", "dev_m_alt", "dev_m_roll", "m_theta1", "m_theta2", "m_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

#  VII. Mechnical Correction (arcmin) by Roll and Altitude

In [27]:
params = MountModelParams.from_config({
    "m3_tilt_alt": -2.1048,
    "m3_tilt_az": +1.5241,
    "m2_tilt_alt_amp": +67.11,
    "m2_tilt_alt_zero": 0.0,
    "m3_encoder_scale": 0.0   
})

for alt in range(70,-1,-10):
    s = f"**Altitude {alt:2.0f}°**   | "
    for roll in range(-70, +71, 10):
        q = q_from_azaltroll(180,alt,roll)
        q_adj, mag = apply_mechanical_corrections(q,params)
        mag = mag*60
        s = s + f"{mag:5.0f} | "
    print(s)


**Altitude 70°**   |   366 |    39 |    40 |    42 |    45 |    50 |    56 |    63 |    71 |    79 |    87 |    94 |   101 |   106 |   473 | 
**Altitude 60°**   |    43 |    40 |    37 |    34 |    35 |    39 |    47 |    58 |    71 |    84 |    97 |   109 |   119 |   127 |   133 | 
**Altitude 50°**   |    59 |    53 |    46 |    38 |    30 |    28 |    36 |    51 |    70 |    89 |   108 |   124 |   138 |   149 |   157 | 
**Altitude 40°**   |    80 |    73 |    64 |    52 |    38 |    23 |    23 |    43 |    70 |    96 |   120 |   141 |   158 |   171 |   181 | 
**Altitude 30°**   |   103 |    97 |    87 |    75 |    59 |    37 |    13 |    34 |    72 |   107 |   137 |   161 |   180 |   195 |   206 | 
**Altitude 20°**   |   128 |   122 |   114 |   103 |    89 |    67 |    32 |    23 |    80 |   126 |   160 |   185 |   205 |   220 |   231 | 
**Altitude 10°**   |   154 |   149 |   143 |   136 |   127 |   113 |    79 |    12 |   111 |   162 |   192 |   215 |   233 |   247 |   257 | 
**Alti

# Scrap Area


In [50]:
fig = px.scatter(d, x="p_theta3", y="dev_p_az", color="p_theta2")
fig.show()